In [1]:
import os
import sys
import json
import random
import shutil
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


Using device: cuda


In [3]:
def download_dataset() -> Path:
    import kagglehub
    path = kagglehub.dataset_download(
        "era2730/forward-looking-sonar-marine-debris-dataset"
    )
    print("Path to dataset files:", path)
    return Path(path)


def explore_tree(root: Path, max_depth: int = 3, max_items: int = 15):
    """Print a bounded tree so we can see the real layout without
    flooding stdout on large datasets."""
    root = Path(root)
    print(f"\n--- Exploring: {root} ---")
    for dirpath, dirnames, filenames in os.walk(root):
        depth = len(Path(dirpath).relative_to(root).parts)
        if depth > max_depth:
            dirnames[:] = []
            continue
        indent = "  " * depth
        print(f"{indent}{Path(dirpath).name}/  "
              f"({len(dirnames)} dirs, {len(filenames)} files)")
        for f in filenames[:5]:
            print(f"{indent}  - {f}")
        if len(filenames) > 5:
            print(f"{indent}  ... +{len(filenames)-5} more files")


IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}


def find_image_folder_root(root: Path):
    """
    Auto-detect an ImageFolder-style structure: a directory whose
    immediate subdirectories each contain images (these subdirs are
    treated as class labels). Searches a few levels deep because
    Kaggle zips often nest an extra folder or two.
    """
    candidates = []
    for dirpath, dirnames, filenames in os.walk(root):
        dirpath = Path(dirpath)
        if not dirnames:
            continue
        class_like = 0
        for d in dirnames:
            sub = dirpath / d
            has_imgs = any(
                p.suffix.lower() in IMG_EXTS for p in sub.iterdir()
                if p.is_file()
            )
            if has_imgs:
                class_like += 1
        if class_like >= 2:  # at least 2 "classes" with images
            candidates.append((dirpath, class_like))

    if not candidates:
        return None
    # Prefer the shallowest match with the most class-like subfolders
    candidates.sort(key=lambda x: (len(x[0].parts), -x[1]))
    return candidates[0][0]


In [4]:
class SonarImageDataset(Dataset):
    """
    Generic ImageFolder-style dataset for sonar images.
    Expects: root/class_a/*.png, root/class_b/*.png, ...
    Loads as single-channel (grayscale) since sonar imagery is
    intensity-based, not RGB.
    """

    def __init__(self, samples, class_to_idx, transform=None):
        self.samples = samples  # list of (filepath, label_idx)
        self.class_to_idx = class_to_idx
        self.transform = transform

    @classmethod
    def from_folder(cls, root: Path, transform=None):
        root = Path(root)
        classes = sorted(
            [d.name for d in root.iterdir() if d.is_dir()]
        )
        class_to_idx = {c: i for i, c in enumerate(classes)}
        samples = []
        for c in classes:
            for p in (root / c).rglob("*"):
                if p.is_file() and p.suffix.lower() in IMG_EXTS:
                    samples.append((str(p), class_to_idx[c]))
        return cls(samples, class_to_idx, transform)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("L")  # grayscale
        if self.transform:
            img = self.transform(img)
        return img, label


In [13]:
IMG_SIZE = 224
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.RandomRotation(15),

    transforms.RandomAffine(
        degrees=0,
        translate=(0.08, 0.08),
        scale=(0.9, 1.1)
    ),

    transforms.ColorJitter(
        brightness=0.25,
        contrast=0.25
    ),

    transforms.RandomResizedCrop(
        IMG_SIZE,
        scale=(0.8, 1.0)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.5],
        std=[0.5]
    ),

    transforms.Lambda(
        lambda x: x.repeat(3, 1, 1)
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
    transforms.Lambda(lambda x: x.repeat(3, 1, 1)),
])

In [14]:
def build_model(num_classes: int, freeze_backbone: bool = True):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

    if freeze_backbone:
      for param in model.parameters():
          param.requires_grad = False

      for param in model.layer3.parameters():
          param.requires_grad = True

      for param in model.layer4.parameters():
          param.requires_grad = True

    in_features = model.fc.in_features
    model.fc = nn.Sequential(
    nn.Linear(in_features, 512),
    nn.BatchNorm1d(512),
    nn.ReLU(),
    nn.Dropout(0.5),

    nn.Linear(512, 256),
    nn.BatchNorm1d(256),
    nn.ReLU(),
    nn.Dropout(0.4),

    nn.Linear(256, 128),
    nn.BatchNorm1d(128),
    nn.ReLU(),
    nn.Dropout(0.3),

    nn.Linear(128, num_classes)
)
    return model.to(DEVICE)


In [18]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    torch.set_grad_enabled(is_train)
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

        if is_train:
            optimizer.zero_grad()

        outputs = model(imgs)
        loss = criterion(outputs, labels)

        if is_train:
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * imgs.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    torch.set_grad_enabled(True)
    avg_loss = total_loss / total
    acc = correct / total
    return avg_loss, acc, all_preds, all_labels


def train_model(model, train_loader, val_loader, class_names,
                 epochs=30, lr=1e-3, patience=7, out_dir="outputs"):
    os.makedirs(out_dir, exist_ok=True)

    # class weighting to handle imbalance (common in debris datasets)
    labels = [l for _, l in train_loader.dataset.dataset.samples] \
        if isinstance(train_loader.dataset, torch.utils.data.Subset) \
        else [l for _, l in train_loader.dataset.samples]
    counts = Counter(labels)
    weights = torch.tensor(
        [1.0 / counts[i] for i in range(len(class_names))],
        dtype=torch.float32,
    ).to(DEVICE)

    ccriterion = nn.CrossEntropyLoss(
    weight=weights,
    label_smoothing=0.1
)
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=1e-4,
    )
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=3
    )

    best_val_loss = float("inf")
    epochs_no_improve = 0
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc, _, _ = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc, _, _ = run_epoch(model, val_loader, criterion, None)
        scheduler.step(val_loss)

        history["train_loss"].append(tr_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(tr_acc)
        history["val_acc"].append(val_acc)

        print(f"Epoch {epoch:02d}/{epochs} | "
              f"train_loss={tr_loss:.4f} train_acc={tr_acc:.3f} | "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.3f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            torch.save({
                "model_state": model.state_dict(),
                "class_names": class_names,
                "img_size": IMG_SIZE,
            }, os.path.join(out_dir, "best_model.pt"))
            print("  -> saved new best checkpoint")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {epoch} "
                      f"(no val improvement for {patience} epochs)")
                break

    with open(os.path.join(out_dir, "history.json"), "w") as f:
        json.dump(history, f, indent=2)

    plot_history(history, out_dir)
    return history


def plot_history(history, out_dir):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(history["train_loss"], label="train")
    axes[0].plot(history["val_loss"], label="val")
    axes[0].set_title("Loss"); axes[0].legend()
    axes[1].plot(history["train_acc"], label="train")
    axes[1].plot(history["val_acc"], label="val")
    axes[1].set_title("Accuracy"); axes[1].legend()
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "training_curves.png"), dpi=150)
    plt.close(fig)

In [20]:
def evaluate_model(model, test_loader, class_names, out_dir="outputs"):
    criterion = nn.CrossEntropyLoss()
    _, acc, preds, labels = run_epoch(model, test_loader, criterion, None)
    print(f"\nTest accuracy: {acc:.4f}\n")

    report = classification_report(
        labels, preds, target_names=class_names, zero_division=0
    )
    print(report)
    with open(os.path.join(out_dir, "classification_report.txt"), "w") as f:
        f.write(report)

    cm = confusion_matrix(labels, preds)
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(class_names)))
    ax.set_yticks(range(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha="right")
    ax.set_yticklabels(class_names)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                     color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title("Confusion Matrix")
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "confusion_matrix.png"), dpi=150)
    plt.close(fig)

In [21]:
def main(explore_only: bool = False, freeze_backbone: bool = True,
         epochs: int = 30, batch_size: int = 32, out_dir: str = "outputs"):

    raw_root = download_dataset()
    explore_tree(raw_root)

    data_root = find_image_folder_root(raw_root)
    if data_root is None:
        print("\n[!] Could not auto-detect an ImageFolder-style layout "
              "(root/class_name/*.png). Inspect the tree above and set "
              "`data_root` manually, or adapt SonarImageDataset to your "
              "annotation format (e.g. COCO-style bounding boxes need a "
              "detection pipeline, not this classifier).")
        return
    print(f"\nDetected class-folder root: {data_root}")

    if explore_only:
        classes = sorted([d.name for d in data_root.iterdir() if d.is_dir()])
        print(f"Detected classes ({len(classes)}): {classes}")
        for c in classes:
            n = len(list((data_root / c).glob("*")))
            print(f"  {c}: {n} files")
        print("\nEXPLORE_ONLY=True — stopping before training. "
              "Re-run with explore_only=False once this looks right.")
        return

    full_dataset = SonarImageDataset.from_folder(data_root)
    class_names = sorted(full_dataset.class_to_idx,
                          key=full_dataset.class_to_idx.get)
    print(f"Classes: {class_names}")
    print(f"Total images: {len(full_dataset)}")

    # 70/15/15 split
    n = len(full_dataset)
    n_train = int(0.70 * n)
    n_val = int(0.15 * n)
    n_test = n - n_train - n_val
    train_set, val_set, test_set = random_split(
        full_dataset, [n_train, n_val, n_test],
        generator=torch.Generator().manual_seed(SEED),
    )

    # apply distinct transforms per split
    train_set.dataset = SonarImageDataset(
        full_dataset.samples, full_dataset.class_to_idx, train_transform)
    val_set.dataset = SonarImageDataset(
        full_dataset.samples, full_dataset.class_to_idx, eval_transform)
    test_set.dataset = SonarImageDataset(
        full_dataset.samples, full_dataset.class_to_idx, eval_transform)

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=2)

    model = build_model(num_classes=len(class_names), freeze_backbone=freeze_backbone)

    train_model(model, train_loader, val_loader, class_names,
                epochs=epochs, out_dir=out_dir)

    # reload best checkpoint before final test evaluation
    ckpt = torch.load(os.path.join(out_dir, "best_model.pt"), map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])
    evaluate_model(model, test_loader, class_names, out_dir=out_dir)

    print(f"\nDone. Best checkpoint + reports saved in ./{out_dir}/")


if __name__ == "__main__":
    # STEP 1: run with explore_only=True first to sanity-check the
    # detected folder structure and class names before training.
    main(explore_only=True)

    # STEP 2: once the printed classes/counts look correct, switch to:
    # main(explore_only=False, freeze_backbone=True, epochs=30, batch_size=32)


Using Colab cache for faster access to the 'forward-looking-sonar-marine-debris-dataset' dataset.
Path to dataset files: /kaggle/input/forward-looking-sonar-marine-debris-dataset

--- Exploring: /kaggle/input/forward-looking-sonar-marine-debris-dataset ---
forward-looking-sonar-marine-debris-dataset/  (2 dirs, 0 files)
  marine-fls/  (1 dirs, 0 files)
    marine-debris-fls-datasets-master/  (1 dirs, 1 files)
      - README.md
      md_fls_dataset/  (2 dirs, 2 files)
        - __init__.py
        - loader.py
  marine-debris-fls-datasets-watertank-v1.0/  (1 dirs, 0 files)
    marine-debris-fls-datasets-watertank-v1.0/  (0 dirs, 1 files)
      - README.md

Detected class-folder root: /kaggle/input/forward-looking-sonar-marine-debris-dataset/marine-fls/marine-debris-fls-datasets-master/md_fls_dataset/data/turntable-cropped
Detected classes (18): ['brown-glass-bottle', 'can', 'drink-carton', 'drink-sachet', 'glass-bottle', 'glass-jar', 'large-tire', 'metal-bottle', 'metal-box', 'plastic-bid